In [ ]:
%pip install pandas matplotlib seaborn scikit-learn


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

sns.set_theme(style="whitegrid")


In [ ]:
feature_names = [
    "radius",
    "texture",
    "perimeter",
    "area",
    "smoothness",
    "compactness",
    "concavity",
    "concave_points",
    "symmetry",
    "fractal_dimension",
]

columns = ["id", "diagnosis"]
for prefix in ["mean", "se", "worst"]:
    columns.extend(f"{prefix}_{name}" for name in feature_names)

df = pd.read_csv("wdbc.data", header=None, names=columns)
df.head()


In [ ]:
print(f"Rows: {df.shape[0]}, columns: {df.shape[1]}")
display(df["diagnosis"].value_counts().rename("count"))
display(df.isna().sum().rename("missing_values"))
display(df.describe())


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

sns.countplot(data=df, x="diagnosis", hue="diagnosis", palette="Set2", legend=False, ax=axes[0, 0])
axes[0, 0].set_title("Diagnosis counts")
axes[0, 0].set_xlabel("Diagnosis")
axes[0, 0].set_ylabel("Count")

sns.scatterplot(
    data=df,
    x="mean_radius",
    y="mean_texture",
    hue="diagnosis",
    palette="Set2",
    alpha=0.8,
    ax=axes[0, 1],
)
axes[0, 1].set_title("Mean radius vs. mean texture")

sns.boxplot(data=df, x="diagnosis", y="mean_area", hue="diagnosis", palette="Set2", legend=False, ax=axes[1, 0])
axes[1, 0].set_title("Mean area by diagnosis")

sns.boxplot(data=df, x="diagnosis", y="worst_concave_points", hue="diagnosis", palette="Set2", legend=False, ax=axes[1, 1])
axes[1, 1].set_title("Worst concave points by diagnosis")

plt.tight_layout()
plt.show()


In [ ]:
X = df.drop(columns=["id", "diagnosis"])
y = df["diagnosis"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"Training rows: {len(X_train)}")
print(f"Testing rows: {len(X_test)}")


In [ ]:
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000),
)

model.fit(X_train, y_train)


In [ ]:
y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(classification_report(y_test, y_pred))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues")
plt.title("Diagnosis prediction confusion matrix")
plt.show()


In [ ]:
models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Support Vector Machine": make_pipeline(StandardScaler(), SVC()),
    "K-Nearest Neighbors": make_pipeline(StandardScaler(), KNeighborsClassifier()),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}

results = []
for name, candidate_model in models.items():
    candidate_model.fit(X_train, y_train)
    candidate_predictions = candidate_model.predict(X_test)
    results.append(
        {
            "model": name,
            "accuracy": accuracy_score(y_test, candidate_predictions),
            "precision_malignant": precision_score(y_test, candidate_predictions, pos_label="M"),
            "recall_malignant": recall_score(y_test, candidate_predictions, pos_label="M"),
            "f1_malignant": f1_score(y_test, candidate_predictions, pos_label="M"),
        }
    )

model_results = pd.DataFrame(results).sort_values("recall_malignant", ascending=False)
display(model_results)

plt.figure(figsize=(10, 5))
sns.barplot(
    data=model_results.melt(id_vars="model", value_vars=["accuracy", "recall_malignant", "f1_malignant"]),
    x="value",
    y="model",
    hue="variable",
)
plt.xlim(0.85, 1.0)
plt.title("Model comparison on the test set")
plt.xlabel("Score")
plt.ylabel("Model")
plt.legend(title="Metric")
plt.show()
